# Step 2 候補の測定・llama-server 版（docs/v2-light-model-plan.md §4・Colab T4 用）

追加探索の 2 番目 **Gemma 4 E4B-it（Q4_K_M の GGUF・考え OFF）** を **e5-large・リランク off** で G0 / Q / N / P / V を測る。問・プロンプト・出力上限 1024・温度 0.1・合格の線は第 1・第 2 候補（step2_measure.ipynb の vLLM 版）と同じ。データは架空の社内規程（evaluation/corpus）だけ。トークン不要。ランタイムは **T4 GPU**。

vLLM 版との違い（計画 §4 の 10/23 の決まり。結果と一緒に必ず書く）:

- 10/24 の差し替え: 追加探索の 1 番目 Qwen3.5-4B（6f32981 で測定済み）から重みだけ替えた。手順・線・問は同じ。Q8_0（8.19GB）は 8GB の予算を超えるので Q4_K_M にした（10/24 Sol と決めた順）。考える機能は `--reasoning off` で切る（Gemma 4 E2B と同じ）。起動の確認は測定の起動の GPU の確認と流し読みの事前確認で兼ね、通らなければそこで止まる
- 動かす物は llama.cpp b11836 の配布版 llama-server（`-ngl 999 --jinja --reasoning off -np 1 -v`）。重みは Q4_K_M・会話の形は GGUF 同梱・抜き出しの設定は llama の既定
- llama-server は GPU の割合で先取りしないので、8GB と 12GB は同じ設定（文脈 = 必要な長さ）で起動し、それぞれ実測の最大を自分の予算と比べる
- 起動のたびに GPU の確認（全層が GPU・GPU 側に重み）。通らなければその回は不合格。数えない問を 1 つ先に流す
- 測定の起動の後に 1 回だけ、流し読みで入力トークン数が返る・本文が content に入る・考える文が空かを確かめる。通らなければ止める
- KV の追い出しの代わりに文脈あふれの行を見る。デバッグ行が 1 本も無ければ未判定（通ったことにしない）

流れ（「Run all」で全部流れる。止まるのは準備の失敗だけ）:

1. 準備（仮想環境・Qdrant・GPU の記録・llama-server と重み）→ 索引
2. 測定: 文脈 8192 で起動し、62 問を `LLM_MAX_TOKENS=1` で流して入力トークン数を取る → 文脈 = `ceil((最大 + 1024) / 512) × 512`
3. 予算ごと（8GB → 12GB）: GPU が空くのを待つ → 起動 → 62 問を `--no-judge` で回答
4. 回答の控え（セルの出力）→ G0（12GB の設定）→ まとめ（summary.json）→ zip

読み方:

- 予算ごとの `ok` は全問回答・最長の問・OOM なし・文脈あふれなし・予算内。`None` は「取れなかった＝未確認」で、通ったことにはしない
- 結果は `/content/step2g4e4b/` に残る。最後のセルで zip にする（本文を含むので、リポジトリに入れるのは run_meta.json とまとめだけにする）
- 起動の記録（-v）には問と回答の本文が入るので、セルには拾った行だけを出す


In [ ]:
# RAG 側の仮想環境（step2_measure.ipynb と同じ）。llama-server は配布版を使うので vLLM は入れない
!pip -q install uv
!git clone -q --depth 1 -b redesign/v2 https://github.com/Maaa2005/local-rag-agent.git /content/repo
!uv venv -q /content/rag-env
# torch は CPU 版を先に入れる（同じ行で CPU の index を足すと uv が他の依存を解けない）
!uv pip install -q --python /content/rag-env/bin/python torch --index-url https://download.pytorch.org/whl/cpu
!uv pip install -q --python /content/rag-env/bin/python "transformers<5" sentence-transformers==3.3.1 \
    qdrant-client==1.12.1 aiosqlite==0.20.0 pydantic-settings==2.6.1 openai==1.57.0 httpx==0.28.1 pyyaml

In [ ]:
# 設定
# Step 2 追加探索の 4 番目（§4・10/24 Sol と決めた順）: Gemma 4 E4B-it Q4_K_M・考え OFF に C（数の比べをコードで）を ON。
# 2 番目（f4b493d）とモデル・重み・llama.cpp・文脈の決め方は同じで、違いは C だけ
MODEL = "google/gemma-4-E4B-it"
GGUF_REPO = "unsloth/gemma-4-E4B-it-GGUF"
GGUF_REV = "bfc15c382204943c3a8fff0c750b94ae2364d7a3"
GGUF_FILE = "gemma-4-E4B-it-Q4_K_M.gguf"
QUANT = "Q4_K_M"
LLAMA_TAG = "b11836"
LLAMA_ASSET = f"llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"
CUDART_ASSET = f"cudart-llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"
EXTRA_ARGS = []
# C を ON（backend の settings.numeric_check_enabled）。索引以外のすべての評価・G0 に渡す
NUMERIC_CHECK = "true"
# 確かめ用 48 問（C を作った後に 1 回だけ流す。文書も別の置き場に載せる）
HOLDOUT_DATASET = "evaluation/v2/numeric_holdout48.jsonl"
HOLDOUT_CORPUS = "evaluation/v2/corpus_nx_holdout"
# 採点役は起動しない（10/23 に人手採点へ。回答はセルの出力と zip に残す）
JUDGE_CANDIDATES = []
MEASURE_CTX = 8192                        # 入力トークン数を測る起動（測る前なので長めに取る）
FAULT_TIMEOUT = 15                        # G0 の止める試験で待つ秒数
QDRANT_VER = "v1.12.6"                    # qdrant-client 1.12.1 に合わせる
REPO = "/content/repo"
RAG_PY = "/content/rag-env/bin/python"
OUT = "/content/step2con"                   # 結果の置き場所（やり直すときはセルを流す前に消すか名前を替える）
PORT = 8001
STEP_NOTE = f"Step 2 候補 {MODEL} {QUANT} + C ON（llama.cpp {LLAMA_TAG}・8GB と 12GB は同じ設定で実測を比べる・採点は人手）"

In [ ]:
import json, os, pathlib, re, signal, subprocess, sys, threading, time, urllib.request

sys.path.insert(0, REPO)   # evaluation.colab.step05 を使う（標準ライブラリだけなので kernel から import できる）
from evaluation.colab import step05 as s5

OUTP = pathlib.Path(OUT)
if OUTP.exists() and any(OUTP.iterdir()):
    raise RuntimeError(f"{OUT} が空でない（前の結果と混ざるので止める）。消すか設定セルの OUT を替える")
OUTP.mkdir(parents=True, exist_ok=True)

def launch(argv, log, env=None):
    # 新しいプロセスグループで起動する（落とすときに killpg を使うため。pid = pgid）
    f = open(log, "w")
    return subprocess.Popen(argv, stdout=f, stderr=subprocess.STDOUT, start_new_session=True, env=env)

def wait(url, proc, log, limit):
    # 起動したら True。プロセスが落ちた・時間切れなら、エラーらしい行だけ出して False（止めずに記録へ回す）
    t = time.time()
    while time.time() - t < limit:
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen(url, timeout=3)
            print("ready:", url, f"{time.time() - t:.0f} 秒")
            return True
        except Exception:
            time.sleep(3)
    lines = [l[:400] for l in pathlib.Path(log).read_text(errors="replace").splitlines()]
    hits = [i for i, l in enumerate(lines) if re.search(r"Traceback|ERROR|Error|error|Exception|Killed|failed|out of memory", l)]
    print("--- 起動しない:", url, "（全文は", log, "）---")
    print("\n".join(f"{i + 1}: {lines[i][:300]}" for i in hits[:40]) or "エラーらしい行なし")
    return False

def kill(proc):
    if proc is None:
        return
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    proc.wait()

def read_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return s5.parse_smi_mib(out)

# Qdrant
qd = pathlib.Path("/content/qdrant"); (qd / "bin").mkdir(parents=True, exist_ok=True)
!curl -sSL https://github.com/qdrant/qdrant/releases/download/{QDRANT_VER}/qdrant-x86_64-unknown-linux-gnu.tar.gz | tar xz -C /content/qdrant/bin
q_env = dict(os.environ, QDRANT__STORAGE__STORAGE_PATH=str(qd / "storage"), QDRANT__SERVICE__HTTP_PORT="6333")
qdrant = launch([str(qd / "bin" / "qdrant")], "/content/qdrant.log", q_env)
if not wait("http://127.0.0.1:6333/readyz", qdrant, "/content/qdrant.log", 60):
    raise RuntimeError("Qdrant が起動しない（準備の失敗なのでここで止める）")

# V（GPU メモリ）: カード全体の使用量を 1 秒ごとに記録する（vLLM 版と同じ間隔）
GPU_MEM = []   # (time.time(), MiB)
def _log_gpu_mem():
    while True:
        try:
            GPU_MEM.append((time.time(), read_mib()))
        except Exception:
            pass
        time.sleep(1)
if "GPU_MEM_THREAD" not in globals():
    GPU_MEM_THREAD = threading.Thread(target=_log_gpu_mem, daemon=True)
    GPU_MEM_THREAD.start()

GPU_NAME, CAP, TOTAL_MIB = [x.strip() for x in subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.splitlines()[0].split(",")]
TOTAL_MIB = int(TOTAL_MIB)
BASELINE_MIB = read_mib()   # 何も載せていない時の使用量（GPU が空いたかの基準）

# llama-server（配布版・ubuntu・CUDA 12.8。Step 0.5 で T4 の全層 GPU を確かめた物）
pre = pathlib.Path("/content/llama-pre"); pre.mkdir(exist_ok=True)
base = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}"
for a in (LLAMA_ASSET, CUDART_ASSET):
    r = subprocess.run(["bash", "-c", f"set -o pipefail; curl -sfL {base}/{a} | tar xz -C {pre}"], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{a} を取れない（準備の失敗なのでここで止める）: {r.stderr[-300:]}")
found = sorted(pre.rglob("llama-server"))
if not found:
    raise RuntimeError("配布版に llama-server が無い（準備の失敗なのでここで止める）")
SERVER_BIN = str(found[0])
libdirs = sorted({str(p.parent) for p in pre.rglob("*.so*")})
LLAMA_ENV = dict(os.environ, LD_LIBRARY_PATH=":".join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]))
v = subprocess.run([SERVER_BIN, "--version"], capture_output=True, text=True, env=LLAMA_ENV)
SERVER_VER = (v.stdout + v.stderr).strip().splitlines()[-2:] if (v.stdout + v.stderr).strip() else ["不明"]

# 重み（revision 固定・トークン不要）
from huggingface_hub import hf_hub_download
GGUF_PATH = hf_hub_download(GGUF_REPO, GGUF_FILE, revision=GGUF_REV, local_dir="/content/gguf")
REV = {"sha": GGUF_REV, "now": s5.hf_revision(GGUF_REPO).get("sha")}
print(GPU_NAME, "compute", CAP, "全体", TOTAL_MIB, "MiB / 起動前", BASELINE_MIB, "MiB")
print("llama-server:", SERVER_VER, "/ GGUF", GGUF_FILE, f"{os.path.getsize(GGUF_PATH) / 2**30:.2f} GiB", "/ revision", REV)

In [ ]:
# 1. 索引（別プロセス・llama-server の前）。DATA_DIR・QDRANT_URL・QDRANT_COLLECTION は index.json に残り、評価側はそこから作る
WORK = OUTP / "index_work"
COLLECTION = f"step2con_{int(time.time())}"
p = subprocess.run([RAG_PY, f"{REPO}/evaluation/colab/corpus_index.py", "--repo", REPO, "--work", str(WORK),
                    "--qdrant-url", "http://127.0.0.1:6333", "--collection", COLLECTION],
                   cwd=REPO, capture_output=True, text=True)
print(p.stdout[-2000:], p.stderr[-3000:])
if p.returncode != 0:
    raise RuntimeError("索引に失敗（準備の失敗なのでここで止める）")
INDEX = json.loads((WORK / "index.json").read_text())
print(INDEX)

In [ ]:
# 2. 測定: 入力トークン数だけ取る（LLM_MAX_TOKENS=1 はこのプロセスだけ）
BASE_URL = f"http://127.0.0.1:{PORT}/v1"
WINDOWS = {}    # 区間名 -> (開始, 終了)。GPU の記録の切り出しに使う

def run_eval(argv, env, name, log):
    t0 = time.time()
    with open(log, "w") as f:
        rc = subprocess.run(argv, cwd=REPO, env=dict(os.environ, **env), stdout=f, stderr=subprocess.STDOUT).returncode
    WINDOWS[name] = (t0, time.time())
    tail = pathlib.Path(log).read_text(errors="replace").splitlines()[-8:]
    print(f"[{name}] rc={rc} {WINDOWS[name][1] - t0:.0f} 秒\n  " + "\n  ".join(tail))
    return rc

def start_server(ctx, log):
    # 起動 → GPU の確認 → 数えない問を 1 つ。返り値は (proc, 起動したか, GPU の確認, 数えない問の秒)
    proc = launch(s5.llama_server_argv(SERVER_BIN, GGUF_PATH, PORT, ctx, extra=EXTRA_ARGS), str(log), LLAMA_ENV)
    up = wait(f"http://127.0.0.1:{PORT}/health", proc, str(log), 600)
    gpu = s5.llama_gpu_check(pathlib.Path(log).read_text(errors="replace"))
    warm = None
    if up and gpu["ok"]:
        body = json.dumps({"model": "local-llm", "messages": [{"role": "user", "content": "こんにちは"}],
                           "temperature": 0.1, "max_tokens": 16}).encode()
        t = time.time()
        try:
            req = urllib.request.Request(f"{BASE_URL}/chat/completions", body, {"Content-Type": "application/json"})
            urllib.request.urlopen(req, timeout=300).read()
            warm = round(time.time() - t, 1)
        except Exception as e:
            warm = f"失敗: {repr(e)[:200]}"
    print(f"[{pathlib.Path(log).name}] 起動 {up} / GPU {gpu} / 数えない問 {warm}")
    return proc, up, gpu, warm

def c_record_counts(raw_path, ids=None):
    # C の記録（足したか・保留の理由）と結論の一致を数えるだけ。問・回答・文書の本文は読まない
    from collections import Counter
    rows = [json.loads(l) for l in pathlib.Path(raw_path).read_text(encoding="utf-8").splitlines() if l.strip()]
    rows = [r for r in rows if ids is None or r["id"] in ids]
    rec = [r.get("numeric_check") for r in rows]
    out = {"n": len(rows), "recorded": sum(x is not None for x in rec), "fired": sum(bool(x and x.get("fired")) for x in rec),
           "hold_reasons": dict(Counter(x.get("hold_reason") for x in rec if x and not x.get("fired")))}
    graded = [r for r in rows if r.get("expected_conclusion") is not None]
    if graded:
        out.update(graded=len(graded), conclusion_ok=sum(r.get("conclusion_ok") is True for r in graded),
                   errors=sum(bool(r.get("error")) for r in graded),
                   not_ok_ids=[r["id"] for r in graded if r.get("conclusion_ok") is not True],
                   fired_not_ok_ids=[r["id"] for r in graded if r.get("conclusion_ok") is not True
                                     and (r.get("numeric_check") or {}).get("fired")])
    return out

# 流し読みの事前確認（評価と同じ openai の呼び方。本文は出さず数だけ）
PRECHECK_PY = r'''
import json, re, sys
from openai import OpenAI
c = OpenAI(base_url=sys.argv[1], api_key="NONE")
s = c.chat.completions.create(model="local-llm", messages=[{"role": "user", "content": "日本の首都はどこですか。一文で答えてください。"}],
                              max_tokens=64, temperature=0.1, stream=True, stream_options={"include_usage": True})
content, reasoning, usage = "", "", None
for ch in s:
    if getattr(ch, "usage", None) is not None:
        usage = ch.usage
    if ch.choices:
        d = ch.choices[0].delta
        content += d.content or ""
        reasoning += ((getattr(d, "model_extra", None) or {}).get("reasoning_content") or "")
print(json.dumps({"prompt_tokens": getattr(usage, "prompt_tokens", None), "content_chars": len(content),
                  "reasoning_chars": len(reasoning), "japanese": bool(re.search(r"[\u3040-\u30ff]", content)),
                  "thinking_marks": bool(re.search(r"<\|?channel|<\|?think", content))}))
'''

MEASURE = {"ctx": MEASURE_CTX}
s5.wait_gpu_released(read_mib, BASELINE_MIB)
srv, up, gpu, warm = start_server(MEASURE_CTX, OUTP / "llama_measure.log")
if not (up and gpu["ok"]):
    raise RuntimeError("測定用の llama-server が起動しない・GPU に全層が載らない（文脈の長さが決まらないので止める）")
pc = subprocess.run([RAG_PY, "-c", PRECHECK_PY, BASE_URL], capture_output=True, text=True)
try:
    PRECHECK = json.loads(pc.stdout.strip().splitlines()[-1])
except Exception:
    PRECHECK = {"error": (pc.stdout + pc.stderr)[-500:]}
PRECHECK["ok"] = bool((PRECHECK.get("prompt_tokens") or 0) > 0 and PRECHECK.get("content_chars", 0) > 0
                      and PRECHECK.get("reasoning_chars") == 0 and not PRECHECK.get("thinking_marks"))
print("PRECHECK_JSON " + json.dumps(PRECHECK, ensure_ascii=False))
if not PRECHECK["ok"]:
    kill(srv)
    raise RuntimeError("流し読みの事前確認が通らない（入力トークン数・本文・考える文のどれか）。測っても線と比べられないので止める")
META_COMMON = {"step": "2", "gpu": GPU_NAME, "compute_cap": CAP, "model": MODEL, "model_revision": GGUF_REV,
               "runtime": f"llama.cpp {LLAMA_TAG}", "gguf": f"{GGUF_REPO}/{GGUF_FILE}", "quant": QUANT,
               "rerank": "off", "numeric_check": "on", "note": STEP_NOTE}
rc = run_eval(s5.eval_cmd(RAG_PY, OUTP / "measure", meta={**META_COMMON, "purpose": "prompt_tokens",
                                                          "ctx": MEASURE_CTX, "llm_max_tokens": 1}),
              s5.eval_env(INDEX["env"], BASE_URL, LLM_MAX_TOKENS="1", NUMERIC_CHECK_ENABLED=NUMERIC_CHECK), "measure", str(OUTP / "measure.log"))
kill(srv)
STATS = s5.prompt_token_stats(OUTP / "measure" / "raw.jsonl")   # 1 問でも取れなければ止まる
C_ON = c_record_counts(OUTP / "measure" / "raw.jsonl")
print("C_MEASURE_JSON " + json.dumps(C_ON, ensure_ascii=False))
if C_ON["recorded"] != C_ON["n"]:
    raise RuntimeError("C の記録が無い問がある（C が ON になっていない）。比べられないので止める")
MAX_LEN = s5.needed_max_model_len(STATS["max_prompt_tokens"])
MEASURE.update(STATS, rc=rc, needed_ctx=MAX_LEN, gpu_check=gpu, warmup_seconds=warm, precheck=PRECHECK, numeric_check=C_ON)
print("入力の最大", STATS["max_prompt_tokens"], "トークン（", STATS["max_id"], "・", STATS["n"], "問）→ 文脈", MAX_LEN)
if MAX_LEN > MEASURE_CTX:
    print("注意: 必要な長さが測定の起動（", MEASURE_CTX, "）より長い。測定は通ったので値は使えるが、回答の起動も足りない可能性がある")

In [ ]:
# 3. 予算ごと（8GB → 12GB）に同じ設定（文脈 = 必要な長さ）で起動して 62 問を全部。実測の最大を各予算と比べる
ALL_IDS = s5.dataset_ids(REPO)   # 重複があれば止まる
RUNS = {}   # "8GB" / "12GB" -> 結果

def run_budget(budget):
    budget_mib = s5.BUDGETS_MIB[budget]
    rec = {"budget": budget, "budget_mib": budget_mib, "gpu_util": None, "ctx": MAX_LEN}
    s5.wait_gpu_released(read_mib, BASELINE_MIB)
    log = OUTP / f"llama_{budget}.log"
    t_launch = time.time()
    proc, up, gpu, warm = start_server(MAX_LEN, log)
    WINDOWS[f"{budget}:launch"] = (t_launch, time.time())
    rec.update(gpu_check=gpu, warmup_seconds=warm)
    if not (up and gpu["ok"]):
        kill(proc)
        rec.update(started=up, gpu_ok=gpu["ok"], peak_mib=s5.peak_in_window(GPU_MEM, t_launch, time.time())["peak_mib"],
                   checks={"ok": False, "note": "起動しない・GPU に全層が載らない"})
        RUNS[budget] = rec
        print(f"[{budget}] 起動しない・GPU に全層が載らない。不合格として記録して先へ進む")
        return rec
    log_start = len(log.read_text(errors="replace"))
    out_dir = OUTP / f"run_{budget}"
    rc = run_eval(s5.eval_cmd(RAG_PY, out_dir, meta={**META_COMMON, "budget": budget, "budget_mib": budget_mib,
                                                     "ctx": MAX_LEN}),
                  s5.eval_env(INDEX["env"], BASE_URL, NUMERIC_CHECK_ENABLED=NUMERIC_CHECK), f"{budget}:run", str(OUTP / f"run_{budget}.log"))
    text = log.read_text(errors="replace")
    kill(proc)
    whole, run_part = s5.scan_llama_log(text), s5.scan_llama_log(text[log_start:])
    # OOM と文脈あふれは起動から全部、デバッグ行の数と文脈の長さは回答の区間だけで見る
    log_scan = dict(whole, debug_active=run_part["debug_active"], slot_n_ctx=run_part["slot_n_ctx"])
    mem = s5.gpu_mem_record(GPU_MEM, {"launch": WINDOWS[f"{budget}:launch"], "run": WINDOWS[f"{budget}:run"]},
                            budget_mib, TOTAL_MIB, BASELINE_MIB)
    checks = s5.trial_checks(out_dir / "raw.jsonl", ALL_IDS, STATS["max_id"], mem,
                             s5.llama_preempt_delta(log_scan), log_scan, note=STEP_NOTE)
    if (out_dir / "run_meta.json").exists():
        s5.attach_to_run_meta(out_dir, "gpu_mem", mem)
        s5.attach_to_run_meta(out_dir, "run_checks", checks)
        s5.attach_to_run_meta(out_dir, "llama_log_scan", log_scan)
        s5.attach_to_run_meta(out_dir, "gpu_check", gpu)
    rec.update(started=True, gpu_ok=True, rc=rc, out_dir=str(out_dir), peak_mib=mem["peak_mib"], checks=checks,
               log_scan=log_scan)
    RUNS[budget] = rec
    print(f"[{budget}] 最大 {mem['peak_mib']} MiB / 予算 {budget_mib} MiB → ok={checks['ok']}"
          f" / デバッグ行 {log_scan['debug_active']} / 文脈 {log_scan['slot_n_ctx']} / OOM {len(log_scan['oom'])} / あふれ {len(log_scan['preempt'])}")
    # P（時間）を問ごとにセルの出力へ全部出す（本文は出さない。ランタイムが切れても数値が残る）
    if (out_dir / "raw.jsonl").exists():
        nums = s5.numbers_only(out_dir / "raw.jsonl")
        rec["times"] = nums["times"]
        rec["numeric_check"] = c_record_counts(out_dir / "raw.jsonl")
        rec["numeric_check_N"] = c_record_counts(out_dir / "raw.jsonl", {i for i in ALL_IDS if i.startswith("N")})
        print(f"C_{budget}_JSON " + json.dumps({"all": rec["numeric_check"], "N": rec["numeric_check_N"]}, ensure_ascii=False))
        print(f"P_{budget}_JSON " + json.dumps({"budget": budget, "checks": checks, "peak_mib": mem["peak_mib"],
                                                "gpu_check": gpu, "log_scan": log_scan, **nums}, ensure_ascii=False))
    return rec

for budget in s5.BUDGETS_MIB:
    run_budget(budget)


In [ ]:
# 3.2 確かめ用 48 問（C を作った後に 1 回だけ。§4 の 10/24「測る前に決めた」C の決まり）
# 主の測定と同じモデル・重み・設定で、別の文書を別の集合に載せて流す。8GB と 12GB は同じ設定なので 1 回だけ。
# C ON を先に、その後に同じ起動のまま C OFF（比べるための対照）。出すのは結論の一致と C の記録の数だけ（本文は出さない）
HOLDOUT = {"dataset": HOLDOUT_DATASET, "corpus": HOLDOUT_CORPUS, "budget_note": "8GB と 12GB は同じ設定なので 1 回だけ"}
H_RUNS = {}   # "holdout_on" / "holdout_off" -> 出力の置き場
def run_quiet(argv, env, name, log):
    # run_eval と同じだが、記録の末尾（問の頭が入る）を出さず rc と秒だけ出す
    t0 = time.time()
    with open(log, "w") as f:
        rc = subprocess.run(argv, cwd=REPO, env=dict(os.environ, **env), stdout=f, stderr=subprocess.STDOUT).returncode
    WINDOWS[name] = (t0, time.time())
    print(f"[{name}] rc={rc} {WINDOWS[name][1] - t0:.0f} 秒")
    return rc
srv = None
try:
    H_IDS = s5.dataset_ids(REPO, datasets=(HOLDOUT_DATASET,))
    WORK_H = OUTP / "holdout_index_work"
    p = subprocess.run([RAG_PY, f"{REPO}/evaluation/colab/corpus_index.py", "--repo", REPO, "--work", str(WORK_H),
                        "--qdrant-url", "http://127.0.0.1:6333", "--collection", f"{COLLECTION}_holdout",
                        "--corpus", f"{REPO}/{HOLDOUT_CORPUS}"], cwd=REPO, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"確かめ用の索引に失敗: rc={p.returncode}")
    H_INDEX = json.loads((WORK_H / "index.json").read_text())
    HOLDOUT["index"] = {k: H_INDEX.get(k) for k in ("documents", "chunks", "seconds")}
    H_META = {**META_COMMON, "purpose": "holdout48", "dataset": HOLDOUT_DATASET}
    # 入力の長さ（回答は 1 トークンだけ・C ON）→ 文脈は主の測定と同じか、足りなければ長くする
    s5.wait_gpu_released(read_mib, BASELINE_MIB)
    srv, up, gpu, warm = start_server(MEASURE_CTX, OUTP / "llama_holdout_measure.log")
    if not (up and gpu["ok"]):
        raise RuntimeError("確かめ用の測定の llama-server が起動しない")
    run_quiet(s5.eval_cmd(RAG_PY, OUTP / "holdout_measure", meta={**H_META, "ctx": MEASURE_CTX, "llm_max_tokens": 1},
                         datasets=(HOLDOUT_DATASET,)),
             s5.eval_env(H_INDEX["env"], BASE_URL, LLM_MAX_TOKENS="1", NUMERIC_CHECK_ENABLED=NUMERIC_CHECK),
             "holdout:measure", str(OUTP / "holdout_measure.log"))
    kill(srv); srv = None
    H_STATS = s5.prompt_token_stats(OUTP / "holdout_measure" / "raw.jsonl")
    H_LEN = max(MAX_LEN, s5.needed_max_model_len(H_STATS["max_prompt_tokens"]))
    HOLDOUT.update(max_prompt_tokens=H_STATS["max_prompt_tokens"], ctx=H_LEN)
    s5.wait_gpu_released(read_mib, BASELINE_MIB)
    srv, up, gpu, warm = start_server(H_LEN, OUTP / "llama_holdout.log")
    if not (up and gpu["ok"]):
        raise RuntimeError("確かめ用の llama-server が起動しない")
    for name, flag in (("holdout_on", NUMERIC_CHECK), ("holdout_off", "false")):
        out_dir = OUTP / name
        rc = run_quiet(s5.eval_cmd(RAG_PY, out_dir, meta={**H_META, "ctx": H_LEN, "numeric_check": "on" if flag == "true" else "off"},
                                  datasets=(HOLDOUT_DATASET,)),
                      s5.eval_env(H_INDEX["env"], BASE_URL, NUMERIC_CHECK_ENABLED=flag), name, str(OUTP / f"{name}.log"))
        H_RUNS[name] = str(out_dir)
        HOLDOUT[name] = dict(c_record_counts(out_dir / "raw.jsonl"), rc=rc) if (out_dir / "raw.jsonl").exists() else {"rc": rc, "missing": True}
        got = {json.loads(l)["id"] for l in (out_dir / "raw.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()} if (out_dir / "raw.jsonl").exists() else set()
        HOLDOUT[name]["all_ids"] = got == set(H_IDS)
except Exception as e:
    HOLDOUT["error"] = repr(e)[:500]   # 止めずに記録し、G0 とまとめへ進む（決まり: 終わる前の失敗だけやり直してよい）
finally:
    kill(srv)
print("HOLDOUT_JSON " + json.dumps(HOLDOUT, ensure_ascii=False))


In [ ]:
# 3.5 回答の控え: 各予算の raw.jsonl・report.md・run_meta.json を gzip+base64 でセルの出力に出す。
# ランタイムが切れても、人手採点の用紙をこの回答から作れる（中身は架空の規程に対する回答）
import base64, gzip, io, tarfile
buf = io.BytesIO()
with tarfile.open(fileobj=buf, mode="w") as tar:
    for label, rec in RUNS.items():
        d = pathlib.Path(rec.get("out_dir", ""))
        for name in ("raw.jsonl", "report.md", "run_meta.json"):
            if rec.get("out_dir") and (d / name).exists():
                tar.add(d / name, arcname=f"run_{label}/{name}")
blob = base64.b64encode(gzip.compress(buf.getvalue())).decode()
print(f"ANSWERS_B64_LEN {len(blob)}")
print("ANSWERS_B64_BEGIN")
for i in range(0, len(blob), 10000):
    print(blob[i:i + 10000])
print("ANSWERS_B64_END")


In [ ]:
# 4. 採点: このノートでは採点しない（10/23 に人手採点へ切り替え。用紙は回答をそろえてから作る）
JUDGE = {"role": "human", "status": "未実施（回答だけ集める）"}
JUDGED = {}
print("JUDGE_JSON " + json.dumps(JUDGE, ensure_ascii=False))

In [ ]:
# 5. G0（RAG 接続の確認・リランク off・12GB と同じ設定）。止める試験で両サーバーを落とすので最後に流す
import shutil
G0 = {"rerank": "off", "fault_timeout": FAULT_TIMEOUT, "numeric_check": NUMERIC_CHECK}
if RUNS.get("12GB", {}).get("started") and RUNS["12GB"].get("gpu_ok"):
    s5.wait_gpu_released(read_mib, BASELINE_MIB)
    G0.update(budget="12GB", ctx=MAX_LEN)
    srv, up, gpu, warm = start_server(MAX_LEN, OUTP / "llama_g0.log")
    G0["gpu_check"] = gpu
    if up and gpu["ok"]:
        WORK_G0 = OUTP / "g0_work"
        p = subprocess.run([RAG_PY, f"{REPO}/evaluation/colab/rag_connection_check.py", "--repo", REPO,
                            "--work", str(WORK_G0), "--qdrant-url", "http://127.0.0.1:6333", "--vllm-url", BASE_URL,
                            "--qdrant-pgid", str(qdrant.pid), "--vllm-pgid", str(srv.pid),
                            "--fault-timeout", str(FAULT_TIMEOUT), "--rerank", "off",
                            "--label", f"colab-{GPU_NAME.split()[-1]}-step2-llama-rerank-off"],
                           cwd=REPO, capture_output=True, text=True,
                           env=dict(os.environ, NUMERIC_CHECK_ENABLED=NUMERIC_CHECK))
        (OUTP / "g0.log").write_text(p.stdout + "\n--- stderr ---\n" + p.stderr)
        print(p.stdout[-4000:], p.stderr[-3000:])
        res = WORK_G0 / "rag_check_result.json"
        G0["rc"] = p.returncode
        G0["ok"] = json.loads(res.read_text()).get("ok") if res.exists() else None   # 結果が無ければ未確認
        if res.exists():
            shutil.copy(res, OUTP / "g0_result.json")
    else:
        G0["skipped"] = "G0 用の llama-server が起動しない・GPU に全層が載らない"
    kill(srv)
else:
    G0["skipped"] = "12GB で起動しなかったので G0 は流さない"
print("G0:", G0)


In [ ]:
# 6. まとめ（Step 2 候補・llama-server・C ON）
rows = []
for label, rec in RUNS.items():
    c = rec.get("checks") or {}
    ls = rec.get("log_scan") or {}
    rows.append({"label": label, "ctx": rec["ctx"], "started": rec["started"], "gpu_ok": rec.get("gpu_ok"),
                 "peak_mib": rec.get("peak_mib"), "budget_mib": rec["budget_mib"], **{k: c.get(k) for k in
                 ("all_answered", "longest_ok", "no_oom", "no_preemption", "within_budget", "ok", "missing")},
                 "debug_active": ls.get("debug_active"), "slot_n_ctx": ls.get("slot_n_ctx"),
                 "warmup_seconds": rec.get("warmup_seconds"), "gpu_check": rec.get("gpu_check"), "times": rec.get("times"),
                 "numeric_check": rec.get("numeric_check"), "numeric_check_N": rec.get("numeric_check_N")})
keys = list(rows[0]) if rows else []
print(" | ".join(keys))
for r in rows:
    print(" | ".join(str(r[k]) for k in keys))
SUMMARY = {"step": "2", "note": STEP_NOTE, "gpu": GPU_NAME, "compute_cap": CAP, "total_mib": TOTAL_MIB,
           "baseline_mib": BASELINE_MIB, "runtime": f"llama.cpp {LLAMA_TAG}", "server_version": SERVER_VER,
           "model": MODEL, "gguf": {"repo": GGUF_REPO, "file": GGUF_FILE, "revision": REV}, "quant": QUANT,
           "server_args": s5.llama_server_argv("llama-server", GGUF_FILE, PORT, MAX_LEN, extra=EXTRA_ARGS)[1:],
           "rerank": "off", "numeric_check": NUMERIC_CHECK, "measure": MEASURE, "holdout": HOLDOUT, "n_questions": len(ALL_IDS), "runs": rows, "judge": JUDGE,
           "judged": JUDGED, "g0": G0}
(OUTP / "summary.json").write_text(json.dumps(SUMMARY, ensure_ascii=False, indent=2) + "\n")
print("SUMMARY_JSON " + json.dumps(SUMMARY, ensure_ascii=False))
print("→", OUTP / "summary.json")


In [ ]:
# サーバーを落とし、結果を zip にする（回答の本文を含む。リポジトリに入れるのは run_meta.json と summary.json だけ）
kill(qdrant)
!cd /content && zip -qr step2con_results.zip step2con -x 'step2con/index_work/*' 'step2con/holdout_*' 'step2con/llama_holdout*' 'step2con/g0_work/*'
!ls -la /content/step2con_results.zip